In [29]:
import torch
from transformers import (
    AutoTokenizer,           # loads the tokenizer for any model
    AutoModelForCausalLM,    # loads the actual model weights
    BitsAndBytesConfig,      # configures 4-bit quantization
)
from peft import (
    LoraConfig,              # defines your LoRA adapter settings
    get_peft_model,          # attaches adapters to the frozen model
    TaskType,                # tells LoRA what kind of task this is
)

In [30]:
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU found: {gpu_name}")
    print(f"GPU memory: {gpu_mem:.1f} GB")
    print("You're good to go!")
else:
    print("NO GPU FOUND!")
    print("Go to: Runtime → Change runtime type → T4 GPU")



GPU found: Tesla T4
GPU memory: 15.6 GB
You're good to go!


# **Model Quantization**

In [31]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [32]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # compress weights to 4-bit
    bnb_4bit_quant_type="nf4",               # use NormalFloat4 format
    bnb_4bit_compute_dtype=torch.float16,    # math happens in float16
    bnb_4bit_use_double_quant=True,          # extra compression (saves ~0.4GB more)
)

torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False


print("4-bit quantization config ready!")
print("  Storage format : 4-bit NF4")
print("  Compute format : float16")
print("  Double quant   : enabled")


4-bit quantization config ready!
  Storage format : 4-bit NF4
  Compute format : float16
  Double quant   : enabled


# **Tokenization**

In [33]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [34]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Set pad_token = eos_token")

print(f"Tokenizer loaded!")
print(f"Vocabulary size: {tokenizer.vocab_size:,} tokens")

# Quick demo — see what tokenization looks like
sample = "fix: handle null pointer in auth"
tokens = tokenizer.encode(sample)
print(f"\nExample tokenization:")
print(f"  Text  : '{sample}'")
print(f"  Tokens: {tokens}")
print(f"  Count : {len(tokens)} tokens")



Tokenizer loaded!
Vocabulary size: 32,000 tokens

Example tokenization:
  Text  : 'fix: handle null pointer in auth'
  Tokens: [1, 2329, 29901, 4386, 1870, 4879, 297, 4817]
  Count : 8 tokens


# **Load Model**

In [35]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,   # use our 4-bit config
    device_map="auto",                # put on GPU automatically
)

# Check how much GPU memory we're using now
mem_used = torch.cuda.memory_allocated() / 1e9
print(f"\nModel loaded!")
print(f"GPU memory used: {mem_used:.2f} GB")
print(f"(Full 32-bit would have used ~4.4 GB — we're using much less!)")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")



Model loaded!
GPU memory used: 1.85 GB
(Full 32-bit would have used ~4.4 GB — we're using much less!)
Total parameters: 615,606,272


In [36]:
# Force the entire model to float16 — overrides any bf16 that snuck in
import torch

# Cast all bf16 params to float16
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

# Also fix the input embeddings
model.get_input_embeddings().weight.data = \
    model.get_input_embeddings().weight.data.to(torch.float16)

# Verify
dtypes = {}
for _, p in model.named_parameters():
    t = str(p.dtype)
    dtypes[t] = dtypes.get(t, 0) + 1
print("Dtypes after fix:", dtypes)
# Should show NO torch.bfloat16

Dtypes after fix: {'torch.float16': 47, 'torch.uint8': 154}


# **Prepare model for QLoRA training**

In [37]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
print("Model prepared for QLoRA training!")

Model prepared for QLoRA training!


# **Configure LoRA adapters**

In [38]:
lora_config = LoraConfig(
    r=16,                          # adapter rank (size)
    lora_alpha=32,                 # scaling factor (2x r is standard)
    lora_dropout=0.05,             # dropout for regularization
    bias="none",                   # don't train bias parameters
    task_type=TaskType.CAUSAL_LM,  # we're doing text generation
    target_modules=[               # which layers get adapters
        "q_proj",                  # query projection in attention
        "v_proj",                  # value projection in attention
    ],
)

print("LoRA config created!")
print(f"  Rank (r)        : {lora_config.r}")
print(f"  Alpha           : {lora_config.lora_alpha}")
print(f"  Dropout         : {lora_config.lora_dropout}")
print(f"  Target modules  : {lora_config.target_modules}")


LoRA config created!
  Rank (r)        : 16
  Alpha           : 32
  Dropout         : 0.05
  Target modules  : {'v_proj', 'q_proj'}


# **Attach adapters to the model**

In [39]:
model = get_peft_model(model, lora_config)

print("LoRA adapters attached!")
print()

# This shows exactly how many parameters are trainable vs frozen
model.print_trainable_parameters()

LoRA adapters attached!

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [ ]:
print("Running a quick test generation...")
print("(This uses the BASE model — output will be generic)")
print("-" * 50)

test_prompt = (
    "### Instruction:\n"
    "You are an expert developer. Given the git diff below, "
    "write a concise and descriptive commit message.\n\n"
    "Git diff:\n```\n"
    "- def get_user(id):\n"
    "-     return db.query(id)\n"
    "+ def get_user(id):\n"
    "+     if id is None:\n"
    "+         raise ValueError('id cannot be None')\n"
    "+     return db.query(id)\n"
    "```\n\n"
    "### Response:\n"
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,      # generate up to 50 new tokens
        temperature=0.7,        # controls randomness (0=deterministic, 1=creative)
        do_sample=True,         # use sampling (not greedy)
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(f"Base model output: '{response.strip()}'")
print("-" * 50)
print("(After fine-tuning this will say something like:")
print(" 'fix: add None check to get_user function')")

Running a quick test generation...
(This uses the BASE model — output will be generic)
--------------------------------------------------
Base model output: '```
feat: Implement error handling for get_user()

This commit adds error-handling logic to `get_user()` to handle invalid user ids. If an `id` is not provided, we raise a `Value'
--------------------------------------------------
(After fine-tuning this will say something like:
 'fix: add None check to get_user function')


In [41]:
mem_used      = torch.cuda.memory_allocated() / 1e9
mem_reserved  = torch.cuda.memory_reserved() / 1e9
mem_total     = torch.cuda.get_device_properties(0).total_memory / 1e9
mem_free      = mem_total - mem_reserved

print("\nGPU Memory Summary:")
print(f"  Total available : {mem_total:.1f} GB")
print(f"  Used by model   : {mem_used:.2f} GB")
print(f"  Reserved        : {mem_reserved:.2f} GB")
print(f"  Free for training: {mem_free:.2f} GB")
print()

if mem_free > 4:
    print("Memory looks good! You have enough headroom for training.")
else:
    print("Memory is tight. Consider reducing batch size in Step 6.")

print()
print("=" * 50)
print("Model is loaded and ready.")
print("=" * 50)



GPU Memory Summary:
  Total available : 15.6 GB
  Used by model   : 2.12 GB
  Reserved        : 2.48 GB
  Free for training: 13.16 GB

Memory looks good! You have enough headroom for training.

Model is loaded and ready.


# **Train Model**

In [42]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# **Load dataset**

In [43]:
print("Loading dataset...")

dataset = load_dataset(
    "json",
    data_files={
        "train": "train_new.jsonl",   # 90% of your data — model learns from this
        "val":   "val_new.jsonl",     # 10% of your data — honest progress check
    }
)

print(f"Train examples : {len(dataset['train'])}")
print(f"Val examples   : {len(dataset['val'])}")
print()

# Peek at one example so you know what the model will see
print("Sample training text (first 300 chars):")
print("-" * 50)
print(dataset["train"][0]["text"][:300])
print("-" * 50)


Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Train examples : 671
Val examples   : 75

Sample training text (first 300 chars):
--------------------------------------------------
### Instruction:
You are an expert developer. Given the git diff below, write a concise and descriptive commit message that explains what changed and why.

Git diff:
```
diff --git a/.release-please-manifest.json b/.release-please-manifest.json
--- a/.release-please-manifest.json
+++ b/.release-plea
--------------------------------------------------


# **Configure training settings**

In [44]:
training_args = SFTConfig(
    output_dir="./commit-msg-model_new",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=True,
    bf16=False,
    tf32=False,
    fp16_full_eval=True,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=10,
    report_to="none"
)

print("Training config ready!")
print(f"  Epochs          : {training_args.num_train_epochs}")
print(f"  Batch size      : {training_args.per_device_train_batch_size}")
print(f"  Grad accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate   : {training_args.learning_rate}")

# Estimate training time
steps_per_epoch = len(dataset["train"]) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
total_steps = steps_per_epoch * training_args.num_train_epochs
print(f"\nEstimated total steps : ~{total_steps}")
print(f"Estimated time        : ~{total_steps * 3 // 60} minutes on T4")

Training config ready!
  Epochs          : 3
  Batch size      : 4
  Grad accumulation: 4
  Effective batch : 16
  Learning rate   : 0.0002

Estimated total steps : ~123
Estimated time        : ~6 minutes on T4


# **Create the trainer**

In [45]:
trainer = SFTTrainer(
    model=model,                        # your QLoRA model
    args=training_args,                 # all the settings above
    train_dataset=dataset["train"],     # learn from this
    eval_dataset=dataset["val"],        # evaluate on this
    dataset_text_field="text",
    tokenizer=tokenizer                 # converts text → token numbers
)

print("Trainer created!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:289: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/671 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

Trainer created!


# **Main Step - Train Our Model**

In [46]:
print("Starting training...")
print("=" * 50)

train_result = trainer.train()

print("=" * 50)
print("Training complete!")
print(f"Total steps     : {train_result.global_step}")
print(f"Final train loss: {train_result.training_loss:.4f}")


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss
50,1.023600,0.989071
100,0.972000,0.948111


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Training complete!
Total steps     : 126
Final train loss: 1.0604


# **Save the trained adapter weights**

In [47]:
SAVE_PATH = "./commit-msg-adapter_new"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Adapter saved to: {SAVE_PATH}")
print()

# Show what was saved
import os
files = os.listdir(SAVE_PATH)
print("Saved files:")
for f in files:
    size = os.path.getsize(f"{SAVE_PATH}/{f}") / 1e6
    print(f"  {f:40s} {size:.1f} MB")


Adapter saved to: ./commit-msg-adapter_new

Saved files:
  README.md                                0.0 MB
  tokenizer_config.json                    0.0 MB
  training_args.bin                        0.0 MB
  tokenizer.model                          0.5 MB
  special_tokens_map.json                  0.0 MB
  adapter_config.json                      0.0 MB
  adapter_model.safetensors                9.0 MB
  tokenizer.json                           1.8 MB


In [48]:
def generate_commit_message(diff_text, max_new_tokens=60):
    """
    Given a git diff, generate a commit message.
    This is exactly how you'll use the model in real life.
    """
    prompt = (
        "### Instruction:\n"
        "You are an expert developer. Given the git diff below, "
        "write a concise and descriptive commit message.\n\n"
        f"Git diff:\n```\n{diff_text}\n```\n\n"
        "### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,       # low = more focused/deterministic
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return result.strip().split("\n")[0]  # take only first line


# Test on a few real diffs
test_diffs = [
    # Test 1: adding a null check
    """\
- def get_user(id):
-     return db.query(User, id)
+ def get_user(id):
+     if id is None:
+         raise ValueError("id cannot be None")
+     return db.query(User, id)""",

    # Test 2: adding a timeout
    """\
  response = requests.get(url)
+ response = requests.get(url, timeout=30)""",

    # Test 3: renaming a variable
    """\
- def calc(x, y):
-     tmp = x * y
-     return tmp + x
+ def calculate_total(price, quantity):
+     subtotal = price * quantity
+     return subtotal + price""",
]

print("\nTesting your fine-tuned model:")
print("=" * 50)

for i, diff in enumerate(test_diffs, 1):
    msg = generate_commit_message(diff)
    print(f"\nTest {i}:")
    print(f"  Diff    : {diff.strip()[:80]}...")
    print(f"  Output  : {msg}")

print("\n" + "=" * 50)
print("If outputs look like real commit messages — success!")



Testing your fine-tuned model:

Test 1:
  Diff    : - def get_user(id):
-     return db.query(User, id)
+ def get_user(id):
+     if...
  Output  : fix: fix user query to handle None id

Test 2:
  Diff    : response = requests.get(url)
+ response = requests.get(url, timeout=30)...
  Output  : fix: timeout requests.get() in urllib3

Test 3:
  Diff    : - def calc(x, y):
-     tmp = x * y
-     return tmp + x
+ def calculate_total(p...
  Output  : fix: Fix typo in `calculate_total` docstring.

If outputs look like real commit messages — success!


In [49]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [57]:
!cp -r /content/val_new.jsonl /content/drive/MyDrive/FineTuneModels